# Accounts overview

Lists every account in the active data repo with its opening balance and (if applicable) the most recent rate that the scenario applies to it.

`DATA_DIR` comes from `$ANALYSIS_DATA_DIR`. The scenario is resolved the same way the `sim` CLI resolves it: `$ANALYSIS_SCENARIO` if set, else the name recorded in the data repo's `.current_scenario` (written by `sim load <scenario>`), else `current`.

> **Note:** after running cells, `git status` will show this notebook as modified even with no real changes. Staging it (`git add`) will make it show unmodified again, unless actual changes are present — see [notebooks/README.md](./README.md#git-status-shows-a-notebook-as-modified-after-running-it) for why.


In [ ]:
%load_ext autoreload
%autoreload 2

import os
from pathlib import Path

from IPython.display import display

from analysis.sim.loader import resolve_scenario_name

# Same resolution the CLI uses: $ANALYSIS_SCENARIO, else the data repo's
# .current_scenario, else "current".
SCENARIO_REF = resolve_scenario_name(
    os.environ.get("ANALYSIS_DATA_DIR", "../data/shared/sim"),
    os.environ.get("ANALYSIS_SCENARIO") or None,
)
DATA_DIR = SCENARIO_REF.data_dir
SCENARIO = SCENARIO_REF.name
AS_OF = None  # e.g. '2026-05-31' to pin rate lookups
display(SCENARIO_REF)


## Rich table

In [ ]:
from rich.console import Console
from analysis.sim.accounts import load_accounts_view, render_accounts_table

rows = load_accounts_view(DATA_DIR, SCENARIO, as_of=AS_OF)
Console().print(render_accounts_table(rows, title=f"accounts ({DATA_DIR.name})"))

## DataFrame view

Same data as a pandas DataFrame for filtering, grouping, or export.

In [ ]:
from analysis.sim.accounts import accounts_dataframe

df = accounts_dataframe(DATA_DIR, SCENARIO, as_of=AS_OF)
df

In [ ]:
# Summary: total opening balance per kind.
if not df.empty:
    summary = (
        df.dropna(subset=["opening_balance"])
          .groupby("kind", as_index=False)["opening_balance"]
          .sum()
          .sort_values("opening_balance", ascending=False)
    )
    display(summary)

## Balance history (latest run)

Pick one or more accounts (Cmd/Ctrl-click for multi-select) to plot their
month-end balances from the scenario's most recent run (the `latest run`
shown above).

Assets/liabilities and income/expense are plotted separately below since
they're on very different scales and mean different things.


In [ ]:
import sqlite3
import pandas as pd

if SCENARIO_REF.run_path is None:
    raise FileNotFoundError(
        f"no runs for {SCENARIO!r} in {SCENARIO_REF.runs_dir}; run `sim run`"
    )
RUN_FILE = SCENARIO_REF.run_path

run_con = sqlite3.connect(f"file:{RUN_FILE}?mode=ro", uri=True)
try:
    balances = pd.read_sql(
        "SELECT s.month, a.name AS account, a.kind, ast.balance "
        "FROM account_state ast "
        "JOIN simulation_step s ON s.id = ast.step_id "
        "JOIN account a         ON a.id = ast.account_id "
        "ORDER BY s.month, a.name",
        run_con,
    )
finally:
    run_con.close()

balances["month"] = pd.to_datetime(balances["month"])
balances["balance"] = balances["balance"] / 100  # cents -> dollars

balance_pivot = balances.pivot_table(
    index="month", columns="account", values="balance", aggfunc="last"
)
account_kinds = balances.drop_duplicates("account").set_index("account")["kind"]
print(f"{balance_pivot.shape[1]} accounts, {len(balance_pivot)} months")


In [ ]:
from analysis.sim.accounts_plot import account_balance_history_widget
from IPython.display import display

display(
    account_balance_history_widget(
        balance_pivot,
        account_kinds,
        title=f"Asset balances - {RUN_FILE.stem}",
        kind="asset",
    )
)


### Liability history


In [ ]:
from analysis.sim.accounts_plot import account_balance_history_widget
from IPython.display import display

display(
    account_balance_history_widget(
        balance_pivot,
        account_kinds,
        title=f"Liability balances - {RUN_FILE.stem}",
        kind="liability",
    )
)


### Income history

> *NOTE: Income values here are cummulative totals.*


In [ ]:
from analysis.sim.accounts_plot import account_balance_history_widget
from IPython.display import display

display(
    account_balance_history_widget(
        balance_pivot,
        account_kinds,
        title=f"Income balances - {RUN_FILE.stem}",
        kind="income",
    )
)


### Expense history

> *NOTE: Expense values here are cummulative totals.*

In [ ]:
from analysis.sim.accounts_plot import account_balance_history_widget
from IPython.display import display

display(
    account_balance_history_widget(
        balance_pivot,
        account_kinds,
        title=f"Expense balances - {RUN_FILE.stem}",
        kind="expense",
    )
)
